In [12]:
import csv

## Reading the sales data using CSV module

In [13]:
def read_sales(path):
    with open(path, 'r', newline = '',encoding = 'utf-8-sig') as file:
        reader = csv.DictReader(file)
        rows = list(reader) 
        columns = reader.fieldnames
    print("Number of rows:", len(rows))
    print("Column names:", columns)
    return rows   
sales = read_sales(r"C:\Users\Amanda\OneDrive\Desktop\Day2_sales_Analytics\Data\electronics_sales.csv")
print("\nFirst two records:")
for row in sales[:2]:
    print(row)



Number of rows: 50
Column names: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']

First two records:
{'sale_id': 'S-3001', 'sale_date': '2026-09-01', 'store': 'Westgate', 'product': 'Safaricom Neon Ray', 'quantity': '1', 'unit_price_kes': '5999', 'payment_method': 'Cash'}
{'sale_id': 'S-3002', 'sale_date': '2026-09-02', 'store': 'Westgate', 'product': 'USB-C Charger 25W', 'quantity': '2', 'unit_price_kes': '1499', 'payment_method': 'M-Pesa'}


## Converting numeric data types

In [14]:
def to_float(value):
    if value is None:
        return None
    s = str(value).strip()
    if s == '' or s.lower() == 'nan' or s.lower() == 'null':
        return None
    # Tech Hub Kenya messy typing: "KES 1,200", "1,200.00", "  "
    s = s.replace('KES', '').replace('Ksh', '').replace(',', '').replace('%','').strip()
    try:
        return float(s)
    except:
        return None

def to_int(value):
    f = to_float(value)
    return int(f) if f is not None else None

# Clean the data
for row in sales:
    # use .get() so it doesn't crash if column name is slightly different
    row['unit_price'] = to_float(row.get('unit_price_kes') or row.get('unit_price'))
    row['quantity'] = to_int(row.get('quantity'))
    row['discount'] = to_float(row.get('discount_pct') or row.get('discount') or 0)
    row['total'] = to_float(row.get('total_amount_kes') or row.get('total_amount') or row.get('total'))

print("Cleaned sample:")
print(sales[0])

Cleaned sample:
{'sale_id': 'S-3001', 'sale_date': '2026-09-01', 'store': 'Westgate', 'product': 'Safaricom Neon Ray', 'quantity': 1, 'unit_price_kes': '5999', 'payment_method': 'Cash', 'unit_price': 5999.0, 'discount': 0.0, 'total': None}


## Calculating total revenue plus Revenue by product with dictionaries

In [15]:
total_revenue = 0
revenue_by_product = {}

for row in sales:
    t = row['total']
    # If total is missing, we try to recalculate it if possible
    if t is None:
        if row['unit_price'] is not None and row['quantity'] is not None:
            d = row['discount'] or 0
            t = row['unit_price'] * row['quantity'] * (1 - d/100)
            row['total'] = t # fill it back
        else:
            continue # can't calculate, skip this bad row

    total_revenue += t
    
    product = str(row.get('product_name') or row.get('product') or 'Unknown').strip()
    revenue_by_product[product] = revenue_by_product.get(product, 0) + t

print(f"TOTAL REVENUE: KES {total_revenue:,.2f}")
print("\nRevenue by product:")
for prod, rev in revenue_by_product.items():
    print(f"  {prod}: {rev:,.2f}")

TOTAL REVENUE: KES 1,310,812.00

Revenue by product:
  Safaricom Neon Ray: 11,998.00
  USB-C Charger 25W: 19,487.00
  Lenovo Tab M10: 199,992.00
  Xiaomi Redmi Buds: 25,592.00
  Tecno Spark 20: 161,988.00
  JBL Go 3 Speaker: 49,491.00
  Oraimo FreePods: 19,593.00
  Anker Power Bank 20000: 38,691.00
  Hisense 43in Smart TV: 272,993.00
  Samsung Galaxy A15: 56,997.00
  HP 15 Laptop: 389,994.00
  Infinix Hot 40: 63,996.00


## Top 3 ranked and High value sales (for follow-up call)

In [16]:
# Rank top three products - checklist item
sorted_products = sorted(revenue_by_product.items(), key=lambda x: x[1], reverse=True)
top_3 = sorted_products[:3]

print("\nTOP 3 PRODUCTS:")
for i, (prod, rev) in enumerate(top_3, 1):
    print(f"{i}. {prod} - KES {rev:,.2f}")

# Filter high-value transactions with comprehensions - checklist item
# Tech Hub wants big sales for follow-up call, let's say > 100k
HIGH_VALUE = 100000
high_value_sales = [r for r in sales if (r['total'] or 0) >= HIGH_VALUE]

print(f"\nFound {len(high_value_sales)} high-value sales >= KES {HIGH_VALUE:,} for follow-up")
for r in high_value_sales[:5]: # show first 5
    print(r)


TOP 3 PRODUCTS:
1. HP 15 Laptop - KES 389,994.00
2. Hisense 43in Smart TV - KES 272,993.00
3. Lenovo Tab M10 - KES 199,992.00

Found 3 high-value sales >= KES 100,000 for follow-up
{'sale_id': 'S-3017', 'sale_date': '2026-09-10', 'store': 'Kisumu Mega Plaza', 'product': 'Hisense 43in Smart TV', 'quantity': 4, 'unit_price_kes': '38,999', 'payment_method': 'M-Pesa', 'unit_price': 38999.0, 'discount': 0.0, 'total': 155996.0}
{'sale_id': 'S-3022', 'sale_date': '2026-09-13', 'store': 'Nairobi CBD', 'product': 'HP 15 Laptop', 'quantity': 4, 'unit_price_kes': '64999', 'payment_method': 'M-Pesa', 'unit_price': 64999.0, 'discount': 0.0, 'total': 259996.0}
{'sale_id': 'S-3048', 'sale_date': '2026-09-30', 'store': 'Kisumu Mega Plaza', 'product': 'HP 15 Laptop', 'quantity': 2, 'unit_price_kes': '64999', 'payment_method': 'Card', 'unit_price': 64999.0, 'discount': 0.0, 'total': 129998.0}


## Creating reusable statistical functions

In [17]:
# TASK 7: Reusable stats - no imports, no min()/max() inside minmax

def mean(nums):
    # mean = sum / count, layman's: average
    if not nums:  # if list is empty
        raise ValueError("mean() of empty list")  # marking expects this error
    return sum(nums) / len(nums)  # add all and divide

def median(nums):
    # median = middle value after sorting
    if not nums:
        raise ValueError("median() of empty list")
    s = sorted(nums)  # sort small to big
    mid = len(s) // 2  # find middle position
    if len(s) % 2 == 1:  # if odd length
        return s[mid]  # just return middle one
    else:  # if even length
        # average the two middle numbers
        return (s[mid-1] + s[mid]) / 2

def minimum_maximum(nums):
    # find smallest and biggest without using min() or max()
    if not nums:
        raise ValueError("minimum_maximum() of empty list")
    min_v = nums[0]  # start assuming first is both min and max
    max_v = nums[0]
    for x in nums[1:]:  # walk through rest of list
        if x < min_v:  # if we find smaller
            min_v = x
        if x > max_v:  # if we find bigger
            max_v = x
    return min_v, max_v  # return both as tuple

In [18]:
def to_num(value):
    if value is None:
        return None
    value = value.replace(",","")
    try:
        number = float(value)
        if number.is_integer():
            return int(number)
        return(number)
    except ValueError:
        return None
print(to_num("100"))
print(to_num("unknown"))

100
None


## Verification by Pandas

In [21]:
import csv
def read_sales(path):
    with open(path, 'r', newline='', encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        columns = reader.fieldnames
    print("Number of rows:", len(rows))
    print("Column names:", columns)
    return rows

def parse_number(value):
    try:
        if value is None:
            return None
        s = str(value).strip().replace('KES','').replace(',','').strip()
        if s == '':
            return None
        return float(s)
    except ValueError:
        return None

def calculate_revenue(sales):
    total = 0
    for r in sales:
        qty = parse_number(r['quantity'])
        price = parse_number(r['unit_price_kes'])
        if qty is not None and price is not None:
            total += qty * price
    return total

def revenue_by(sales, key):
    result = {}
    for r in sales:
        k = r[key]
        qty = parse_number(r['quantity'])
        price = parse_number(r['unit_price_kes'])
        if qty is None or price is None:
            continue
        rev = qty * price
        result[k] = result.get(k, 0) + rev
    return result

In [23]:
def verify_with_pandas(path, my_results):
    import pandas as pd
    
    def parse_number_pandas(value):
        try:
            if pd.isna(value):
                return None
            s = str(value).strip().replace('KES','').replace(',','').strip()
            if s == '':
                return None
            return float(s)
        except:
            return None

    df = pd.read_csv(path)
    
    df['clean_qty'] = df['quantity'].apply(parse_number_pandas)
    df['clean_price'] = df['unit_price_kes'].apply(parse_number_pandas)
    df = df.dropna(subset=['clean_qty', 'clean_price'])
    
    df['revenue'] = df['clean_qty'] * df['clean_price']
    
    pandas_total = df['revenue'].sum()
    pandas_by_store = df.groupby('store')['revenue'].sum().to_dict()
    pandas_high = len(df[df['revenue'] > 50000])
    
    print(f"Pandas total: {pandas_total}")
    print(f"My total: {my_results['total_revenue']}")
    print(f"Match total? {abs(pandas_total - my_results['total_revenue']) < 0.01}")
    print("\nBy Store comparison:")
    print(f"Pandas: {pandas_by_store}")
    print(f"Mine: {my_results['revenue_by_store']}")

# HOW TO USE:
my_results = {
    'total_revenue': calculate_revenue(sales),
    'revenue_by_store': revenue_by(sales, 'store')
}
verify_with_pandas(r"C:\Users\Amanda\OneDrive\Desktop\Day2_sales_Analytics\Data\electronics_sales.csv", my_results)

Pandas total: 1310812.0
My total: 1310812.0
Match total? True

By Store comparison:
Pandas: {'Kisumu Mega Plaza': 449678.0, 'Mombasa City Mall': 154082.0, 'Nairobi CBD': 331085.0, 'Nakuru Westside': 139988.0, 'Westgate': 235979.0}
Mine: {'Westgate': 235979.0, 'Kisumu Mega Plaza': 449678.0, 'Nakuru Westside': 139988.0, 'Mombasa City Mall': 154082.0, 'Nairobi CBD': 331085.0}


In [24]:
import os
os.makedirs("../output", exist_ok=True)
print("output folder created!")

output folder created!


In [25]:
with open("../output/sales_report.txt", "w") as f:
    f.write("Day 2 Sales Analytics Report\n")
    f.write("============================\n")
    f.write("Total Revenue: 1310812.0\n")
    f.write("Verification: Manual total matches Pandas total - True\n")
print("sales_report.txt created!")

sales_report.txt created!
